[官网文档](https://langchain-zh.cn/oss/javascript/langchain/middleware/custom)

In [1]:
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import after_model
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from typing import Any


from dotenv import load_dotenv

load_dotenv()

class CustomAgentState(AgentState):
  """扩展agent状态，添加自定义字段"""
  model_call_count: int

@after_model(state_schema=CustomAgentState)
def increment_counter(state: CustomAgentState, runtime: Runtime) -> dict[str, Any]:
  """使用装饰器的after_model钩子 增加调用计数"""
  current_count = state.get("model_call_count", 0)
  return { "model_call_count": current_count + 1 }

agent = create_agent(
  model="deepseek-flash",
  middleware=[increment_counter],
  checkpointer=InMemorySaver()
)

config = { "configurable": { "thread_id": "thread_1" } }


In [2]:
response = agent.invoke(
  {"messages": [HumanMessage("你好")]},
  config
)

print(response)

{'messages': [HumanMessage(content='你好', additional_kwargs={}, response_metadata={}, id='e7f69528-d22b-45d2-aa14-c642a3433cfd'), AIMessage(content='你好！很高兴见到你 😊\n\n有什么我可以帮你的吗？无论是聊天、解答问题、写作、翻译、编程，还是帮你理清思路，都可以告诉我。', additional_kwargs={'refusal': None, 'reasoning_content': 'The user just said "你好" (Hello) in Chinese. I should respond warmly in Chinese and offer to help.'}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 31, 'total_tokens': 91, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 24, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 31}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '253b83b3-f433-4070-b237-1db20dfc252f', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1ec76-75b5-7a50-bac3-

In [3]:
response = agent.invoke(
  {"messages": [HumanMessage("现在几点了")]},
  config
)

print(response)

{'messages': [HumanMessage(content='你好', additional_kwargs={}, response_metadata={}, id='e7f69528-d22b-45d2-aa14-c642a3433cfd'), AIMessage(content='你好！很高兴见到你 😊\n\n有什么我可以帮你的吗？无论是聊天、解答问题、写作、翻译、编程，还是帮你理清思路，都可以告诉我。', additional_kwargs={'refusal': None, 'reasoning_content': 'The user just said "你好" (Hello) in Chinese. I should respond warmly in Chinese and offer to help.'}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 31, 'total_tokens': 91, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 24, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 31}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '253b83b3-f433-4070-b237-1db20dfc252f', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1ec76-75b5-7a50-bac3-

In [10]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def retry_model(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
  for attempt in range(3):
    try:
      return handler(request)
    except Exception as e:
      print(f"重试第{attempt + 1}次, error：{e}")
      if attempt == 2:
        raise
  return None

agent = create_agent(
  # model="deepseek-xxx",
  model="deepseek-flash",
  middleware=[retry_model]
)

response = agent.invoke(
  {"messages": [HumanMessage("你好")]},
  config
)

print(response)


{'messages': [HumanMessage(content='你好', additional_kwargs={}, response_metadata={}, id='da24bf22-f96a-4d00-8fd5-0750508f4d81'), AIMessage(content='你好！很高兴见到你 😊 有什么我可以帮你的吗？无论是聊天、解答问题，还是需要帮忙处理什么事情，都可以告诉我。', additional_kwargs={'refusal': None, 'reasoning_content': 'The user just said "你好" (Hello) in Chinese. I should respond warmly in Chinese.'}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 31, 'total_tokens': 81, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 20, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 31}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '3b562fe4-bd98-4a27-a750-287878310ae1', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1ec8e-30d8-7993-92a8-970a9eccc180-0', tool_call

In [12]:
from langchain.agents.middleware import wrap_model_call
from typing import Callable
from dataclasses import dataclass

@dataclass
class UserContext():
  thinking: bool = False


@wrap_model_call
def dynamic_model_setting(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]):
  thinking = request.runtime.context.thinking
  thinking_type = "enabled" if thinking else "disabled"
  model_settings =  {
    "extra_body": {
      "thinking": {"type": thinking_type}
    } 
  }
  modified_request = request.override(model_settings=model_settings)
  return handler(modified_request)


agent = create_agent(
  model="deepseek-flash",
  middleware=[dynamic_model_setting],
  context_schema=UserContext
)

response = agent.invoke(
  {"messages": [HumanMessage("你好")]},
  # context=UserContext(thinking=False)
  context=UserContext(thinking=True)
)

print(response)

{'messages': [HumanMessage(content='你好', additional_kwargs={}, response_metadata={}, id='a0e3c829-fb23-47e2-b7d2-d3542b2c259b'), AIMessage(content='你好！很高兴见到你 😊 有什么我可以帮你的吗？无论是聊天、解答问题、写作、翻译，还是其他任何需要，都可以告诉我～', additional_kwargs={'refusal': None, 'reasoning_content': 'The user just said "你好" (Hello) in Chinese. I should respond warmly in Chinese and offer to help. Keep it brief and friendly.'}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 31, 'total_tokens': 94, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 30, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 31}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '4b478e89-f5dd-453d-8f01-388f3f994b1b', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01

# 条件跳转

In [15]:
from langgraph.runtime import Runtime
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import after_model, AgentMiddleware, hook_config
from langchain.messages import AIMessage
from typing import Any

class CustomAgentState(AgentState):
  model_call_count: int

class ModelCallLimitMiddleware(AgentMiddleware[CustomAgentState]):
  def __init__(self, max_limit: int = 10):
    super().__init__()
    self.max_limit = max_limit

  def after_model(self, state: CustomAgentState, runtime: Runtime) -> dict[str, Any] | None:
    current_count = state.get("model_call_count", 0)
    return { "model_call_count": current_count + 1 }
  
  @hook_config(can_jump_to="end")
  def before_model(self, state: CustomAgentState, runtime: Runtime) -> dict[str, Any] | None:
    current_count = state.get("model_call_count", 0) + 1
    print(f"当前模型调用次数{current_count}/{self.max_limit}")
    if current_count > self.max_limit:
      return {
        "jump_to": "end",
        "messages": [AIMessage(f"模型调用次数已超过{self.max_limit}")]
      }
    return None
  
# @after_model(state_schema=CustomAgentState)
# def increment_counter(state: CustomAgentState, runtime: Runtime):
#   current_count = state.get("model_call_count", 0)
#   return { "model_call_count": current_count + 1 }

agent = create_agent(
  model="deepseek-flash",
  middleware=[ModelCallLimitMiddleware(max_limit=2)],
  state_schema=CustomAgentState,
  checkpointer=InMemorySaver()
)

config = { "configurable": { "thread_id": "thread_1"} }


In [16]:
response = agent.invoke(
  {"messages": [HumanMessage("你好")]},
  config
)
print(response)

当前模型调用次数1/2
{'messages': [HumanMessage(content='你好', additional_kwargs={}, response_metadata={}, id='76cd924f-58a4-4a7c-9b68-05974c72f5da'), AIMessage(content='你好！很高兴见到你 😊 有什么我可以帮你的吗？无论是聊天、解答问题、写作、翻译，还是其他任何事情，我都很乐意协助你。', additional_kwargs={'refusal': None, 'reasoning_content': 'The user just said "你好" (Hello) in Chinese. I should respond warmly in Chinese and offer to help.'}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 31, 'total_tokens': 90, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 24, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 31}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': 'f22ef017-0293-420d-9570-4ecd833d5d7b', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1ecb7-9d42-77

In [17]:
response = agent.invoke(
  {"messages": [HumanMessage("简点说两句")]},
  config
)
print(response)

当前模型调用次数2/2
{'messages': [HumanMessage(content='你好', additional_kwargs={}, response_metadata={}, id='76cd924f-58a4-4a7c-9b68-05974c72f5da'), AIMessage(content='你好！很高兴见到你 😊 有什么我可以帮你的吗？无论是聊天、解答问题、写作、翻译，还是其他任何事情，我都很乐意协助你。', additional_kwargs={'refusal': None, 'reasoning_content': 'The user just said "你好" (Hello) in Chinese. I should respond warmly in Chinese and offer to help.'}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 31, 'total_tokens': 90, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 24, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 31}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': 'f22ef017-0293-420d-9570-4ecd833d5d7b', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1ecb7-9d42-77

In [18]:
response = agent.invoke(
  {"messages": [HumanMessage("还在吗")]},
  config
)
print(response)

当前模型调用次数3/2
{'messages': [HumanMessage(content='你好', additional_kwargs={}, response_metadata={}, id='76cd924f-58a4-4a7c-9b68-05974c72f5da'), AIMessage(content='你好！很高兴见到你 😊 有什么我可以帮你的吗？无论是聊天、解答问题、写作、翻译，还是其他任何事情，我都很乐意协助你。', additional_kwargs={'refusal': None, 'reasoning_content': 'The user just said "你好" (Hello) in Chinese. I should respond warmly in Chinese and offer to help.'}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 31, 'total_tokens': 90, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 24, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 31}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': 'f22ef017-0293-420d-9570-4ecd833d5d7b', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1ecb7-9d42-77